<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-08-agents-and-adk/lesson-8.2-multi-agent/notebooks/GCP_Capstone_8.2_MultiAgent.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 8.2 Multi-Agent Orchestration
**Netsetos GenAI Engineering — GCP Capstone**

Sub-agents, SequentialAgent, ParallelAgent, LoopAgent, AgentTool.


In [ ]:
!pip install -q google-adk google-genai
import os
from google.colab import auth
auth.authenticate_user()
os.environ['GOOGLE_CLOUD_PROJECT'] = 'YOUR-PROJECT'
os.environ['GOOGLE_CLOUD_LOCATION'] = 'us-central1'
os.environ['GOOGLE_GENAI_USE_VERTEXAI'] = 'TRUE'
print('ADK ready (Vertex + ADC)')


## Cell 1: Define Tools


In [ ]:
from google.adk.tools import ToolContext

def search_documents(query: str) -> dict:
    """Search documents by keyword.
    Args:
        query: Search query.
    """
    return {'results': [{'id': 'D-01', 'title': 'Q1 Report'}]}

def summarize_document(document_id: str, summary_type: str) -> dict:
    """Summarize a document.
    Args:
        document_id: Document ID.
        summary_type: brief/detailed/executive.
    """
    return {'summary': 'Summary of ' + document_id}

def extract_entities(text: str) -> dict:
    """Extract named entities from text.
    Args:
        text: Text to analyze.
    """
    return {'entities': {'people': ['John'], 'orgs': ['Acme']}}

def calculate_cost(page_count: int, tier: str) -> dict:
    """Calculate processing cost.
    Args:
        page_count: Pages.
        tier: standard/premium/enterprise.
    """
    rates = {'standard': 0.01, 'premium': 0.03, 'enterprise': 0.05}
    return {'cost_usd': round(page_count * rates.get(tier, 0.01), 2)}

def store_document(doc_id: str, summary: str) -> dict:
    """Store processed document.
    Args:
        doc_id: Document ID.
        summary: Document summary.
    """
    return {'status': 'stored', 'doc_id': doc_id}

print('5 tools defined')


## Cell 2: Sub-Agents with LLM Routing


In [ ]:
from google.adk.agents import LlmAgent

search_agent = LlmAgent(
    name='SearchAgent', model='gemini-3.6-flash',
    description='Searches documents by keyword or metadata.',
    instruction='You are the search specialist. Find relevant documents.',
    tools=[search_documents])

analysis_agent = LlmAgent(
    name='AnalysisAgent', model='gemini-3.6-flash',
    description='Analyzes documents: summaries, entities, sentiment.',
    instruction='You are the analysis specialist. Provide thorough analysis.',
    tools=[summarize_document, extract_entities])

cost_agent = LlmAgent(
    name='CostAgent', model='gemini-3.6-flash',
    description='Calculates processing costs and pricing.',
    instruction='You are the pricing specialist. Calculate costs.',
    tools=[calculate_cost])

root = LlmAgent(
    name='DocuMind', model='gemini-3.6-flash',
    instruction='Route search to SearchAgent, analysis to AnalysisAgent, pricing to CostAgent.',
    sub_agents=[search_agent, analysis_agent, cost_agent])

print('Root agent with 3 sub-agents created')


## Cell 3: SequentialAgent Pipeline


In [ ]:
from google.adk.agents import SequentialAgent

extractor = LlmAgent(
    name='Extractor', model='gemini-3.6-flash',
    instruction='Extract text and metadata from the document provided.',
    output_key='extracted_text')

classifier = LlmAgent(
    name='Classifier', model='gemini-3.6-flash',
    instruction='Classify this document: {extracted_text}\nDetermine category and sensitivity.',
    output_key='classification')

pipeline = SequentialAgent(
    name='IngestionPipeline',
    sub_agents=[extractor, classifier])

print('Sequential pipeline: Extract -> Classify')


## Cell 4: ParallelAgent Fan-Out


In [ ]:
from google.adk.agents import ParallelAgent

legal = LlmAgent(name='Legal', model='gemini-3.6-flash',
    instruction='Analyze for legal risks: {doc_text}',
    output_key='legal_review')

financial = LlmAgent(name='Financial', model='gemini-3.6-flash',
    instruction='Analyze financial impact: {doc_text}',
    output_key='financial_review')

parallel = ParallelAgent(
    name='ParallelReview',
    sub_agents=[legal, financial])

synth = LlmAgent(name='Synthesizer', model='gemini-3.6-flash',
    instruction='Combine reviews: Legal={legal_review} Financial={financial_review}',
    output_key='unified_review')

review_workflow = SequentialAgent(
    name='ReviewWorkflow',
    sub_agents=[parallel, synth])

print('Parallel review + synthesis workflow created')


## Cell 5: LoopAgent Refinement


In [ ]:
from google.adk.agents import LoopAgent

def exit_loop(tool_context: ToolContext):
    """Call when quality is satisfactory."""
    tool_context.actions.escalate = True
    return {'status': 'Refinement complete'}

critic = LlmAgent(name='Critic', model='gemini-3.6-flash',
    instruction='Review this summary: {draft}. If excellent say APPROVED. Else give fixes.',
    output_key='feedback')

refiner = LlmAgent(name='Refiner', model='gemini-3.6-flash',
    instruction='Feedback: {feedback}. Draft: {draft}. If APPROVED call exit_loop. Else improve.',
    tools=[exit_loop],
    output_key='draft')

refinement = LoopAgent(
    name='RefinementLoop',
    sub_agents=[critic, refiner],
    max_iterations=3)

print('LoopAgent: Critic -> Refiner -> repeat (max 3)')


## Cell 6: AgentTool — Workflow as Tool


In [ ]:
from google.adk.tools.agent_tool import AgentTool

# Wrap workflows as callable tools
pipeline_tool = AgentTool(agent=pipeline)
review_tool = AgentTool(agent=review_workflow)

# Root with BOTH sub_agents AND AgentTools
full_root = LlmAgent(
    name='DocuMind', model='gemini-3.6-flash',
    instruction='Route to specialists for queries. Use IngestionPipeline for new docs. Use ReviewWorkflow for deep analysis.',
    sub_agents=[search_agent, analysis_agent, cost_agent],
    tools=[pipeline_tool, review_tool])

print('Full DocuMind: 3 sub-agents + 2 workflow tools')


## Cell 7: Test Multi-Agent System


In [ ]:
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai.types import Content, Part

async def test_multi():
    ss = InMemorySessionService()
    runner = Runner(agent=full_root, app_name='dm', session_service=ss)
    session = await ss.create_session(app_name='dm', user_id='s1')
    
    queries = [
        'Find documents about quarterly revenue',
        'Summarize document D-01 in detail',
        'How much would it cost to process 200 pages at premium tier?',
    ]
    for q in queries:
        print('\nUser: ' + q)
        msg = Content(role='user', parts=[Part(text=q)])
        async for ev in runner.run_async(
            user_id='s1', session_id=session.id, new_message=msg):
            if ev.is_final_response() and ev.content and ev.content.parts:
                for p in ev.content.parts:
                    if p.text: print('Agent: ' + p.text[:150])

await test_multi()


## Done!
- sub_agents for LLM-driven routing
- SequentialAgent for pipelines
- ParallelAgent for concurrent analysis
- LoopAgent for iterative refinement
- AgentTool for workflow invocation
- Session state connects everything
